# Open Azul NWB File

This notebook opens an NWB behavior file with the local `nwb_obj.py` helper.

In [1]:
from pathlib import Path

from nwb_obj import session_data_nwb

nwb_file = Path(r"Y:/User_folders/Azul/behavior_data/M1/20260611/132105_Data/Azul_M1_132105_Data.nwb")
nwb_file

WindowsPath('Y:/User_folders/Azul/behavior_data/M1/20260611/132105_Data/Azul_M1_132105_Data.nwb')

In [2]:
if not nwb_file.exists():
    raise FileNotFoundError(f"NWB file not found: {nwb_file}")

session = session_data_nwb(nwbfile=nwb_file)
session.read_paths()

print(f"Loaded: {session.nwb_path}")
print(f"Found {len(session.ds_paths)} dataset paths")

Loaded: Y:\User_folders\Azul\behavior_data\M1\20260611\132105_Data\Azul_M1_132105_Data.nwb
Found 41 dataset paths


In [3]:
session.ds_paths[:20]

['/acquisition/IRFork/data',
 '/acquisition/IRFork/starting_time',
 '/acquisition/Parameters/key',
 '/acquisition/Parameters/value',
 '/acquisition/Reward/data',
 '/acquisition/Reward/starting_time',
 '/acquisition/TrialType/data',
 '/acquisition/TrialType/starting_time',
 '/file_create_date',
 '/identifier',
 '/intervals/trials/HMCF',
 '/intervals/trials/id',
 '/intervals/trials/sound_ids',
 '/intervals/trials/trial_type',
 '/session_description',
 '/session_start_time',
 '/specifications/core/2.9.0/namespace',
 '/specifications/core/2.9.0/nwb.base',
 '/specifications/core/2.9.0/nwb.behavior',
 '/specifications/core/2.9.0/nwb.device']

In [4]:
session.get_parameters()
session.parameters

{'date': '2026-06-11',
 np.str_('User'): np.str_('Azul'),
 np.str_('Mouse'): np.str_('1'),
 np.str_('Project'): np.str_('Chrono'),
 np.str_('Output'): np.str_('NWB'),
 np.str_('Device'): np.str_('Dev1'),
 np.str_('Channels'): np.str_('ai6,ai5,ai1'),
 np.str_('frec'): np.str_('1000.0'),
 np.str_('TaskType'): np.str_('Lever'),
 np.str_('TriggerType'): np.str_('IRFork'),
 np.str_('Threshold'): np.str_('2.55'),
 np.str_('LeverHoldTime_s'): np.str_('0.7'),
 np.str_('LeverStartDebounce_s'): np.str_('0.1'),
 np.str_('LeverReleaseWindow_s'): np.str_('0.25'),
 np.str_('LeverRequireRelease'): np.str_('1'),
 np.str_('SampleSoundId'): np.str_('1'),
 np.str_('TestSoundId'): np.str_('1'),
 np.str_('DMTSRandomMatchTrials'): np.str_('0'),
 np.str_('DMTSSoundIds'): np.str_('1:16'),
 np.str_('Delay_s'): np.str_('0'),
 np.str_('DMTSForkGrace_s'): np.str_('0.1'),
 np.str_('SoundDuration_s'): np.str_('0'),
 np.str_('ResponseWindow_s'): np.str_('2'),
 np.str_('Rewardduration_ms'): np.str_('150'),
 np.str_('

In [5]:
session.generate_results_table()
session.results_table.head()

,TrialsId,SoundId,TrialType,Hit,Miss,CR,FA
0,1,1,0,1,0,0,0
1,2,1,0,0,1,0,0
2,3,1,0,1,0,0,0
3,4,1,0,1,0,0,0
4,5,1,0,0,1,0,0


In [6]:
session.extract_continuous_signal()

print("sound_signal:", session.sound_signal.shape)
print("whichSound:", session.whichSound.shape)
print("reward_signal:", session.reward_signal.shape)
print("trialtype:", session.trialtype.shape)
print("IR_signal:", session.IR_signal.shape)

sound_signal: (3606600,)
whichSound: (3606600,)
reward_signal: (3606600,)
trialtype: (36067,)
IR_signal: (3606600,)


## IR Signal Viewer

In [7]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import clear_output, display

ir_signal = np.asarray(session.IR_signal).squeeze()
if ir_signal.ndim > 1:
    ir_signal = ir_signal[:, 0]

sampling_rate_hz = None
with h5py.File(session.nwb_path, "r") as h5f:
    starting_time = h5f["acquisition"]["IRFork"].get("starting_time")
    if starting_time is not None and "rate" in starting_time.attrs:
        sampling_rate_hz = float(starting_time.attrs["rate"])

if sampling_rate_hz is None:
    sampling_rate_hz = float(session.parameters.get("frec", 1000.0))

n_samples = len(ir_signal)
duration_s = n_samples / sampling_rate_hz
x_values = np.arange(n_samples) / sampling_rate_hz

window_seconds = widgets.BoundedFloatText(
    value=min(10.0, duration_s),
    min=0.1,
    max=max(duration_s, 0.1),
    step=0.5,
    description="Window (s)",
    layout=widgets.Layout(width="220px"),
)
start_seconds = widgets.FloatSlider(
    value=0,
    min=0,
    max=max(duration_s - window_seconds.value, 0),
    step=0.1,
    description="Start (s)",
    continuous_update=False,
    layout=widgets.Layout(width="700px"),
)

previous_button = widgets.Button(description="Previous", icon="arrow-left")
next_button = widgets.Button(description="Next", icon="arrow-right")
output = widgets.Output()

def _update_start_range(*_):
    start_seconds.max = max(duration_s - window_seconds.value, 0)
    start_seconds.value = min(start_seconds.value, start_seconds.max)

def _bounds():
    start_s = start_seconds.value
    end_s = min(start_s + window_seconds.value, duration_s)
    i0 = int(start_s * sampling_rate_hz)
    i1 = int(end_s * sampling_rate_hz)
    return i0, max(i1, i0 + 1), start_s, end_s

def plot_ir_signal(*_):
    _update_start_range()
    i0, i1, start_s, end_s = _bounds()
    with output:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(x_values[i0:i1], ir_signal[i0:i1], lw=1)
        ax.set_title(f"IR signal: {start_s:.2f} to {end_s:.2f} s")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Amplitude")
        ax.grid(True, alpha=0.3)
        plt.show()

def move_window(direction):
    step = window_seconds.value * 0.8
    start_seconds.value = min(max(start_seconds.value + direction * step, start_seconds.min), start_seconds.max)

previous_button.on_click(lambda _: move_window(-1))
next_button.on_click(lambda _: move_window(1))
start_seconds.observe(plot_ir_signal, names="value")
window_seconds.observe(plot_ir_signal, names="value")

display(widgets.HBox([previous_button, next_button, window_seconds]), start_seconds, output)
plot_ir_signal()

FloatSlider(value=0.0, continuous_update=False, description='Start (s)', layout=Layout(width='600px'), max=360…

FloatSlider(value=10.0, continuous_update=False, description='Window (s)', layout=Layout(width='600px'), max=1…

Output()